


## Autores

- López González Pamela Citlai Atletl


# Práctica 6 Generación de un Entorno de Simulacion en Gazebo

## Objetivo

El objetivo de esta práctica es que el alumno tenga la capacidad de generar entornos de simulación en Gazebo con el fin de que puedan realizar experimentos y pruebas virtuales. 

### Metas 

- Que el alumno aprenda a editar archivos Simulation Data File (.world) en plataformna de Gazebo.
- Crear un paquete *bringup* en el cual se ejecute una simulación donce un robot determinado realice alguna tarea.
  


### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para modelar, simular e interpretar el comportamiento de los sistemas mecatrónicos.
- Para desarrollar, operar y mantener procesos productivos que impliquen la transformación de materia, energía e información.
- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.

#### Actitudes

- Ser creativo e innovador.
- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### De tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.



# Previo de la Práctica 6

Se le solicita al alumno investigar y colocar la sigueinte información en esta sección:

- Investigar los simuladores que son compatibles con ROS 2.
   
    * Gazebo sim

        Ofrece una simulación física robusta mediante motores como ODE y Bullet, soporte avanzado de sensores (cámaras, LiDAR, IMU) y una integración directa a través de paquetes oficiales como ros_gzb2. Es ideal para robótica de servicios, manipulación y entornos de exteriores.

    * Webots

        Simulador de código abierto ampliamente compatible con ROS 2 a través del paquete webots_ros2. Destaca por su ligereza, rapidez de ejecución y una extensa librería de modelos de robots comerciales listos para usar, lo que reduce drásticamente el tiempo de configuración inicial.

    * Isaac Sim
    
        Construido sobre la plataforma Omniverse, es un simulador de alto rendimiento diseñado para aprovechar la aceleración por hardware de las tarjetas gráficas NVIDIA. Proporciona simulaciones fotorrealistas y físicas de ultra-alta fidelidad, orientadas principalmente al entrenamiento de algoritmos de Inteligencia Artificial, Aprendizaje por Refuerzo y visión computacional en ROS 2.

- Investigar cuales son las restricciones del simulador Gazebo 2.

    * Dependencia Estricta del Reloj de Simulación

        Para que los nodos de ROS 2 se sincronicen con la física del simulador, es obligatorio activar el parámetro use_sim_time: True. Si hay retrasos en el procesamiento de la computadora, el tiempo simulado se desfasa del tiempo real, provocando errores críticos de transformación de coordenadas (TF).

    * Complejidad en la Definición de Inercias

        A diferencia de visualizadores como RViz, Gazebo no ignora las propiedades físicas. Si las etiquetas <inertial>, <mass> o las matrices de inercia (ixx, iyy, izz) de un eslabón en el archivo URDF/Xacro están mal calculadas, tienen valores en cero o son físicamente incoherentes con la geometría, el motor de física fallará: el robot colapsará, saldrá expulsión de la escena o simplemente no responderá a los comandos de los controladores de trayectoria.
    
    * Necesidad de un Puente de Comunicación Extrínseco

        En las nuevas versiones de Gazebo, el simulador corre de forma totalmente independiente a ROS 2 utilizando su propio sistema de transporte de mensajes. Para mover el robot o leer sensores, es obligatorio programar y ejecutar un nodo intermedio (un bridge) que traduzca explícitamente los tópicos de Gazebo a tópicos de ROS 2 (como el controlador joint_trajectory que usas en tu código de Python).


## Desarrollo de la práctica 

Para el desarrollo de la práctica se deben realizar las siguientes activiadades:

1. Explicar las relaciones de los archivos xacros.

    Haciendo un simil con el cuerpo humano:
    
    El esqueleto físico es el archivo scara_urdf.xacro, el cual es la base de todo, define la apariencia visual del robot, sus dimensiones reales y dónde colisiona si choca contra algo  Es el archivo puramente estructural.
    
    Los músculos y el movimiento son el archivo gz2_scara.xacro, ya que este archivo contiene los plugins de simulación (JointPositionController y JointStatePublisher). Es el que toma las articulaciones que se crearon en el esqueleto (link_1_joint, link_2_joint, etc.) y les asigna los controladores de posición y las ganancias PID (P, I, D) para que el robot se pueda mover en Gazebo.

    Los ojos son el archivo camera_sensor.xacro, los cuales añaden el eslabón físico de la cámara (camera_link), lo une a la punta del robot y configura el sensor óptico para transmitir el video.

2. Decribir las partes del archivo xacro de la configuración de las juntas del robot.

    Eslabón de fijación al entorno (world y world_joint): Crea un eslabón ficticio llamado world y conecta la base real del robot (base_link) mediante una junta de tipo fixed (completamente soldada, sin movimiento) ubicada en el origen matemático (0,0,0).

    Las juntas son 
    
    *   El link_1_joint que conecta la base (base_link) con el primer eslabón (link_1). Gira alrededor del eje Z (axis xyz="0 0 1"). Tiene una elevación sobre la base de 0.225 metros (origin xyz="0 0 0.225").

    * El link_2_joint conecta el primer brazo (link_1) con el segundo (link_2). Está desfasado hacia adelante 0.45 metros en el eje X (origin xyz="0.45 0 0.05").

    * El link_3_joint conecta el segundo brazo (link_2) con el eslabón final (link_3). Al igual que los anteriores, su rotación es sobre el eje Z, lo que le otorga al robot SCARA la capacidad de moverse de forma paralela al plano horizontal.
    
    Ahora, los límites de Operación (<limit>) de caada una de las tres juntas de revolución cuenta con restricciones físicas idénticas para proteger el comportamiento del motor:
    
    * lower="-3.14159" y upper="3.14159": Limita el giro de la articulación a un rango de $\pm180^\circ$ (un círculo completo en radianes).
    
    * velocity="50.0": Velocidad máxima angular permitida en rad/s.
    
    * effort="1000.0": El torque máximo (fuerza del motor) expresado en Newtons-metro.
    
    En el punto Final (P y link_P_joint): Se añade una junta fija a una distancia de 0.25 metros en X sobre el tercer eslabón para representar la punta de la herramienta (TCP - Tool Center Point), lugar exacto donde ocurrirán las operaciones o donde se colgarán herramientas.

3. Describir las relaciones del archivo xacro del sensor de la cámara.

    El archivo camera_sensor.xacro añade capacidades de visión artificial al robot SCARA mediante una combinación de geometría cinemática y plugins de simulación óptica.

    Anclaje Físico y de Coordenadas 

    * La cámara inicia en el eslabón camera_link (un cubo negro de 3x3x3 cm). Se conecta a través de camera_joint (tipo fixed) al punto final P del robot, sobresaliendo 5 cm hacia adelante en el eje X (origin xyz="0.05 0 0").
    
    Configuración y Parámetros Técnicos del Sensor en Gazebo:Dentro de la etiqueta <sensor name="camera" type="camera">, se parametriza la cámara física virtual donde:
    
    * <update_rate>30.0</update_rate>: Captura y transmite video a una velocidad de 30 cuadros por segundo (FPS).
    
    * <horizontal_fov>1.089</horizontal_fov>: El campo de visión horizontal (Field of View) es de aproximadamente $62.4^\circ$ en radianes, determinando la amplitud del lente.
    
    * <image>: Configura una resolución estándar de 640x480 píxeles con formato de color RGB de 8 bits (R8G8B8).
    
    * <clip>: El rango de visión útil está configurado entre 0.05 metros (mínimo de enfoque) y 8.0 metros (máxima distancia de renderizado).
    
    * <topic>camera/image_raw</topic>: Especifica el buzón de mensajes (tópico) donde Gazebo publicará los bytes de la imagen.
    
    * <gz_frame_id>camera_link_optical</gz_frame_id>: Amarra las imágenes al sistema de coordenadas óptico rotado previamente, garantizando la consistencia espacial en ROS 2.

4. Describir el contenido del paquete scara_bringup y del archivo launch.

    El paquete scara_bringup es el paquete organizador o de "arranque" (bringup). Su propósito no es definir la geometría del robot, sino centralizar los mundos virtuales, configuraciones de interfaces y orquestar el inicio simultáneo de todos los programas necesarios para la simulación mediante un archivo Launch de ROS 2.
    
    Localización de Recursos :

    * Utiliza la función get_package_share_directory para ubicar las carpetas de instalación de scara_description (donde está el modelo del robot) y de scara_bringup (donde busca el archivo de configuración del bridge gz_bridge.yaml, el mundo world_config.world y la interfaz visual de RViz scara_config.rviz).

    Sincronización de Tiempo:

    * la variable use_sim_time = {'use_sim_time': True} obliga a todos los nodos de ROS 2 a ignorar el reloj de la computadora y regirse estrictamente por el reloj físico de Gazebo.

    Nodos Orquestados en la Simulación:

    * robot_state_publisher: Toma el archivo estructural .xacro, procesa matemáticamente sus ecuaciones y publica constantemente el estado de las transformaciones de coordenadas tridimensionales (TFs) del robot.

    * gz_sim: Lanza el binario del simulador externo Gazebo Sim (gz sim), cargando de forma automática el mundo configurado y forzando el uso del motor de renderizado ogre.

    * spaw_entity (ros_gz_sim): Es el nodo encargado de "inyectar" el robot virtual dentro del mundo de Gazebo, tomando el modelo procesado desde el tópico robot_description.

    * gz_ros_bridge_node: Arranca el nodo parameter_bridge. Lee el archivo de configuración YAML para abrir los canales de comunicación bidireccionales entre ROS 2 y Gazebo.

    rviz_node (rviz2): Abre la interfaz gráfica RViz2 cargando la configuración personalizada para visualizar los eslabones del robot y el flujo de datos (como el video de la cámara) de forma interactiva.

5. Describir que hace el archivo bridge. 

    Es el archivo de configuración encargado de realizar el mapeo de tópicos, conversión de tipos de datos y control del flujo de información (bidireccional) entre el middleware de ROS 2 y el transporte nativo de Gazebo Sim. Divide sus tareas en canales de entrada (GZ_TO_ROS) para actualizar el reloj, imágenes de la cámara, nubes de puntos del LiDAR y estado de articulaciones en ROS 2; y canales de salida (ROS_TO_GZ) para enviar las consignas de posición angular calculadas por los algoritmos de control hacia los actuadores del robot en el entorno físico simulado.

## Resultados

### Vídeos



### Demostración Experimental en Entorno Virtual (ROS 2 - Gazebo)

A continuación, se presenta la videodemostración en tiempo real donde se valida la trayectoria de ida y vuelta del manipulador SCARA 3R, la sincronización mediante el puente de comunicación y la transmisión de datos de la cámara hacia RViz.

[![Simulación Robot SCARA 3R](https://img.youtube.com/vi/C1gUPNsJcZs/mqdefault.jpg)](https://www.youtube.com/watch?v=C1gUPNsJcZs)




## Conclusiones

A través del script de MATLAB se logró estructurar el comportamiento teórico y dinámico del manipulador SCARA 3R aplicando las ecuaciones de Euler-Lagrange, sentando las bases físicas del robot. Posteriormente, mediante ROS 2, esta teoría matemática se tradujo en trayectorias de movimiento de ida y vuelta.

La infraestructura completa del espacio de trabajo actuó como un sistema nervioso unificado, donde el paquete de descripción definió la anatomía geométrica del robot y el nodo publicador dictó los comandos de posición de manera secuencial. Estos comandos fueron inyectados directamente hacia los actuadores virtuales en Gazebo Sim gracias a la traducción del puente de comunicación, logrando un desplazamiento suave, continuo y coordinado sin oscilaciones inestables.

Finalmente, la integración visual con RViz2 y Gazebo permitió validar el comportamiento dinámico del robot bajo leyes físicas realistas (fricción, inercia y gravedad), cerrando el ciclo de automatización al transmitir en vivo el flujo de datos de la cámara integrada. El resultado final, documentado en el video demostrativo, constituye la prueba técnica de un proceso completo de ingeniería robótica donde la teoría matemática, la programación y el entorno virtual se unieron para dar vida a un sistema autónomo funcional.

## Bibliografía 

[1] Open Robotics, "Gazebo Simulation Documentation," Gazebosim.org, 2026. [En línea]. Disponible en: https://gazebosim.org/docs

[2] Cyberbotics, "Webots ROS 2 interface documentation," Cyberbotics.com, 2025. [En línea]. Disponible en: https://cyberbotics.com/doc/guide/the-user-interface?version=R2021b

[3] NVIDIA Developer, "NVIDIA Isaac Sim Simulators for ROS 2 Overview," NVIDIA Developer, 2026. [En línea]. Disponible en: https://developer.nvidia.com/isaac/sim

[4] J. M. S. T. Art, "ROS 2 Humble con Gazebo Sim: Configuración del Robot State Publisher y Parameter Bridge," YouTube, 2024. [Video en línea]. Disponible en: https://www.youtube.com/watch?v=FEa2diI2qgA